# 「전자부품(배터리팩) 예지보전 AI 데이터셋」 가이드북 요약

KAMP(Korea AI Manufacturing Platform)에서 제공하는 **「전자부품(배터리팩) 예지보전 AI 데이터셋」 분석실습 가이드북**의
핵심 내용을 정리한 노트북입니다. 실제 EDA/모델링에 앞서 데이터의 배경, 구조, 분석 방법론을 이해하기 위한 참고용입니다.

- 원본: `data/Guidebook_전자부품(배터리팩) 예지보전 AI 데이터셋.pdf`
- 출처: 중소벤처기업부, Korea AI Manufacturing Platform(KAMP), 전자부품(배터리팩) 예지보전 AI 데이터셋,
  스마트제조혁신추진단(㈜인터엑스, 네스트필드㈜), 2022.12.23., www.kamp-ai.kr


## 1. 분석요약

| 구분 | 내용 |
|---|---|
| 분석 목적 | 배터리팩 품질에 영향을 미치는 **레이저 용접설비**를 대상으로, AAS 표준기반 제조데이터 수집/저장 체계를 통해 공정 데이터를 수집하고, N-HiTS 알고리즘으로 설비 이상을 예측하여 불량을 검출하는 설비 이상 예측 모델을 구축 |
| 데이터셋 형태 | 분석 변수: RealPower, SetPower, GateOnTime 외 4개 (총 9개 컬럼) / 수집방법: AAS 기반 TSDB 추출 / 확장자: csv |
| 데이터 개수 | 총 1,268,865개, 총량 7.96MB |
| 적용 알고리즘 | **N-HiTS** (Neural Hierarchical Interpolation for Time Series Forecasting), **통계분포 Z-score** 이상탐지 |
| 분석결과 및 시사점 | 공정 데이터를 활용하여 설비 이상 상태를 사전에 예측하는 시스템을 개발, 실제 E공장에 적용하여 생산성·품질 향상에 기여 |


## 2. 분석 배경

### 2.1 공정(설비) 개요

배터리팩 제조공정은 다음 3단계로 구성됩니다.

1. **배터리셀 검사/조립** — 입고된 배터리셀을 OCV, 절연저항, 내부저항 검사로 양/불 판정 후 조립
2. **배터리모듈 조립** — 배터리셀을 모아 모듈로 조립(반자동), **용접은 용접라인에서 자동 진행**, 용접 후 양/불 판정 → BMS 장착
3. **배터리팩 조립** — 배터리모듈을 케이스에 조립, 충·방전 시험 후 최종 납품 (전 과정 작업자 의존)

### 2.2 이슈사항 (Pain Point)

- 설비 이상 발생 시 생산 지연이 불가피하므로, **사전에 이상 조짐을 감지**하여 대비/점검할 수 있어야 피해를 최소화할 수 있음
- 배터리모듈 조립에 사용되는 다수 설비 중 **불량/이상이 빈번한 레이저 용접 설비**를 분석 대상으로 선정
- 용접 가능한 형태로 모듈을 조립하는 과정이 **수작업**이라, 불량 원인이 설비 이상인지 조립 불량인지 판단이 어려움
- 해결 방안: 용접 완료 모듈의 불량 판정 시각·설비 알람 이력을 기록하고, 실시간 공정 데이터와의 상관관계를 분석하여
  정상/비정상 데이터셋 기반의 AI 학습모델로 불량 원인을 추적하고 설비 이상을 예측


## 3. 분석 목표

- 레이저 용접 설비의 공정 데이터를 기반으로 배터리 품질과의 상관성을 분석하여, 설비 이상을 사전에 감지하고
  현장 작업자가 즉시 확인·조치할 수 있는 기반을 마련
- 데이터 수집: 별도의 데이터 수집 PC를 설치해 OPC UA 통신 인터페이스를 개발, AAS 표준기반 체계로 공정 데이터 및
  설비 상태정보를 수집하여 기계학습 데이터셋을 구성
- 기대효과: 공정 데이터·설비 상태·불량 제품의 상관관계를 도출하여 불량 발생 원인 추적 및 설비 이상 예측이
  필요한 제조공정 전반에 적용 가능, 품질검사/관리 엔지니어에게도 활용 가치가 있음


## 4. 제조데이터 소개

### 4.1 데이터 수집 방법

| 항목 | 내용 |
|---|---|
| 제조(설비) 분야 | 전기차용 배터리팩 제조 |
| 제조 공정명 | 배터리팩 조립 공정 |
| 수집 장비 | AAS(Asset Administration Shell) 서버 |
| 수집 기간 | 2022.07.01 ~ 2022.09.30 |
| 수집 주기 | 약 3초(sec) |

- 현장 설비공정 정보모델은 **IEC 63278-1 AAS** 기술로 모델링되며, `AASX Package Explorer`로 뷰/편집 가능
- 현장 데이터는 PLC → 엣지 게이트웨이 → AAS 서버(TSDB 저장) 경로로 수집됨
- PLC/OPC UA로 수집되는 데이터는 `engineering.csv`에 태그 형태로 매핑됨 (AAS 태그명, 게이트웨이명, 필드장비명,
  OPC UA 태그명, Sampling Interval(ms), 배열 정보, 실제 필드 변수명 등의 컬럼 구성)
- TSDB로부터는 `curl`로 InfluxDB 쿼리를 실행해 CSV를 추출

### 4.2 데이터 유형/구조

- 총 9개 변수로 수집되며, 이 중 **7개 변수**를 실제 분석에 사용 (SetFrequency, SetDuty는 값이 상수라 제거)
- 데이터 개수: 총 1,268,865개 (학습용 1,221,831개 / 테스트용 47,034개)

| 변수 유형 | 변수명 | 설명 |
|---|---|---|
| 용접시퀀스 | PageNo | 용접 작업 시퀀스 정보 (Count) |
| 용접속도설정(mm/s) | Speed | 설정된 길이 기준 모터 속도 설정 |
| 용접길이설정(mm) | Length | 용접할 부분의 길이 설정 |
| 용접출력(W) | **RealPower** | 용접 포인트 별로 측정된 실제 용접 출력 — **핵심 분석 변수** |
| 발광횟수설정(Hz) | SetFrequency | 초당 발광 횟수 설정 (상수, 실습에서 제거) |
| 최대용접출력설정(%) | SetDuty | 최대 용접 출력 설정 (상수, 실습에서 제거) |
| 용접출력설정(%) | SetPower | 용접 재질에 따라 변하는 용접 출력 설정 |
| 용접시간(s) | GateOnTime | 용접 게이트 오픈 시간 |
| 작업시간 | WorkingTime | 작업이 진행된 시간 (타임스탬프) |

독립변수는 위 9개 전부이며, **종속변수는 없음** (비지도 방식의 이상탐지 문제).

### 4.3 주요 변수 기술 통계 (가이드북 기준, Training_Data.csv)

| 구분 | 개수 | 평균 | 표준편차 | 최소값 | 중앙값 | 최대값 |
|---|---|---|---|---|---|---|
| PageNo | 135,759 | 20 | 11.25 | 1 | 20 | 39 |
| Speed | 135,759 | 165.01 | 106.98 | 30 | 250 | 250 |
| Length | 135,759 | 155.53 | 107.81 | 19.4 | 241.1 | 241.2 |
| RealPower | 135,759 | 1310.51 | 493.22 | 0 | 1688 | 1733 |
| SetFrequency | 135,759 | 1000 | 0 | 1000 | 1000 | 1000 |
| SetDuty | 135,759 | 100 | 0 | 100 | 100 | 100 |
| SetPower | 135,759 | 65.38 | 21.65 | 38 | 82 | 83 |
| GateOnTime | 135,759 | 1117.99 | 421.96 | 650 | 1154 | 1670 |

> 실제 다운로드한 `Training_Data.csv`의 shape은 EDA 노트북(`01_eda.ipynb`)에서 직접 확인합니다
> (가이드북 예시와 실데이터의 행 수가 다를 수 있습니다).


## 5. 데이터 품질 전처리

품질이 낮은 데이터로는 좋은 분석 결과를 얻을 수 없으므로, 다음 6가지 품질지수를 기준으로 데이터 품질을 평가하고 개선합니다.

| 품질지수 | 정의 | 계산식 |
|---|---|---|
| 완전성 (Completeness) | 필수 항목에 결측치가 없어야 함 | `1 - 결측데이터수/전체데이터수 × 100` |
| 유일성 (Uniqueness) | 각 엔티티가 고유 식별자로 구분되어야 함 | `유일한데이터수/전체데이터수 × 100` |
| 유효성 (Validity) | 데이터 항목이 정해진 유효범위 안에 있어야 함 | `유효데이터수/전체데이터수 × 100` |
| 일관성 (Consistency) | 데이터의 구조/유형/값이 형태상 일관되어야 함 | `일관데이터수/전체데이터수 × 100` |
| 정확성 (Accuracy) | 실제 개체를 정확히 표현하는 정도 (참조데이터 필요) | `1 - 위배데이터수/전체데이터수 × 100` |
| 무결성 (Integrity) | 유일성·유효성·일관성이 모두 보호되어야 함 | `1 - (100%가 아닌 지수 개수)/3 × 100` |

가이드북 부록 실습 결과, 본 데이터셋은 **완전성·유일성·유효성·일관성·무결성 모두 100%**로 결측치/이상치가
사실상 없는 정제된 데이터셋임을 확인했습니다. (정확성은 참조 데이터가 없어 측정하지 않음)

### 주요 전처리 절차 (가이드북 [단계 ①] 기준)

1. 컬럼명 공백 제거 (`str.strip()`)
2. 단일값(상수)만 갖는 컬럼 제거 → SetFrequency, SetDuty 제거 (9개 → 7개 컬럼)
3. 결측치 확인/처리 (`isna().sum()`, 없으면 스킵 / 있으면 보간)
4. 이상치 탐지: IQR(사분범위) 기반, `Q1 - c×IQR ~ Q3 + c×IQR` 범위를 벗어나면 이상치로 간주 (c=4.0 사용)
5. `RealPower` 값이 1300 미만인 저구간 데이터에 930.5를 더해 두 그룹(저/고 파워) 간 스케일 차이를 줄이는
   보정(학습 안정화 목적, 정보 손실 없음 — 나중에 빼면 원복 가능)


## 6. AI 분석모델

### 6.1 N-HiTS (Neural Hierarchical Interpolation for Time Series Forecasting)

- 다변량 시계열의 미래값을 예측하는 최신 딥러닝 알고리즘 (C. Challu, et al., 2022)
- 핵심 아이디어:
  - **다중-비율 데이터 샘플링(Multi-Rate Data Sampling)**: 시계열을 여러 비율로 부분 샘플링하여 긴 시간/중간 시간/짧은 시간
    스케일의 거동을 각 스택(완전연결 MLP)이 따로 예측
  - **계층적 내삽(Multi-scale Hierarchical Interpolation)**: 부분 샘플링된 예측 결과를 전체 구간에 맞게 보간하여 결합
- 장점: 벤치마크에서 높은 장시간 예측 정확도, 효율적인 메모리 사용, 빠른 계산 속도
- 학습: MAE 손실함수, Adam Optimizer로 최적화 (실습에서는 `MQF2DistributionLoss` 사용)
- 입력/출력 길이 설정 예: `max_encoder_length=20`(입력 구간), `max_prediction_length=10`(예측 구간)

### 6.2 통계분포 Z-score 이상탐지

- N-HiTS는 **비정상 데이터가 연속적으로 긴 구간에 나타나는 경우** 적합하지 않을 수 있음
- 이 경우 학습 데이터의 분포를 여러 개의 가우스 분포 중첩으로 가정하고, 각 모드의 평균/표준편차를 산출
- 테스트 데이터 각 시점 값에 대해 가장 가까운 가우스 모드까지의 거리를 그 모드의 표준편차로 나눈 값(Z-score)을 계산
- Z-score의 최소값이 threshold(예: 4.0)를 넘으면 anomaly로 판정

### 6.3 두 방법의 적용 기준

| 이상 패턴 | 적합 방법 | 대상 테스트 데이터 |
|---|---|---|
| 고립된 시점에서 스파이크성 이상 | N-HiTS (예측값과 실제값 오차의 Z-score) | `WeldingTest_03_NG.csv` |
| 특정 시점 이후 연속 구간 전체가 이상 | 통계분포 Z-score (가우스 혼합) | `WeldingTest_04_NG.csv` |


## 7. 분석 단계별 프로세스 (Flow Chart)

가이드북은 아래 8단계로 분석 실습을 안내합니다.

1. **데이터 준비** — 패키지 설치, 라이브러리 로드, CSV 로드, 컬럼/결측치/이상치 확인, 전처리
2. **데이터 구조 및 특징 탐색** — RealPower가 SetPower(38%/82%/83%)에 따라 주기적 기본 거동 + 잡음 신호를 보임을 확인.
   이상적 RealPower ≈ `2000 × SetPower`이며, 정상 범위는 이상값 대비 ±3~4%(약 ±60~80) 이내
3. **학습/테스트 데이터 선택** — 학습: `Training_Data.csv` (정상, 충분히 긴 구간) / 테스트: 정상 2종 + 비정상 2종
4. **AI 모델 알고리즘 선택** — N-HiTS(고립형 이상) + 통계 Z-score(연속형 이상) 병행 적용
5. **AI 모델 학습** — `TimeSeriesDataSet`으로 학습/검증 데이터 구성 → `NHiTS.from_dataset()`으로 네트워크 정의 →
   최적 학습률 탐색(`lr_find`) → `Trainer.fit()`으로 학습 (max_epochs=30, learning_rate=0.04)
6. **테스트셋에 대한 계산 수행** — 학습된 모델로 테스트 데이터 예측, 실제값과의 오차 계산
7. **테스트 데이터 결과 분석** — 오차 분포의 Z-score를 threshold(4.0)와 비교하여 이상 구간 판정, 실제 라벨과 비교
8. **알고리즘 평가** — Accuracy, Precision, Recall, F1-score 산출


## 8. 분석 결과 요약

| 테스트 데이터 | 이상 패턴 | 적용 방법 | Accuracy | Precision | Recall | F1-score |
|---|---|---|---|---|---|---|
| `WeldingTest_01_OK.csv` | 정상 (이상 없음) | N-HiTS + Z-score(threshold=4.0) | 이상 미검출(정상 판정) | - | - | - |
| `WeldingTest_03_NG.csv` | 고립된 스파이크성 이상 | N-HiTS + Z-score(threshold=4.0) | 0.9960 | 0.8947 | 0.8947 | 0.8947 |
| `WeldingTest_04_NG.csv` | 연속 구간 이상 | 통계분포(가우스 혼합) Z-score(threshold=4.0) | 1.0 | 1.0 | 1.0 | 1.0 |

- N-HiTS 알고리즘은 배터리 용접 공정의 RealPower 시간 변화를 높은 정확도로 예측
- 연속 구간 이상(WeldingTest_04_NG)에는 통계적 분석(가우스 혼합 + Z-score)이 완벽에 가까운 성능을 보임
- 두 방법을 이상 패턴 특성에 맞게 병행 적용하면 높은 정밀도·재현율로 설비 이상을 탐지할 수 있음


## 9. 시사점 및 타 현장 적용 시 고려사항

- 추가 센서 설치 없이 **현장에서 이미 수집 가능한 공정 데이터**만으로 설비 상태를 판별할 수 있다는 점에서 확장성이 큼
- 실제 E공장에 적용하여 생산성 향상과 품질 향상에 기여
- 타 현장 적용 시 고려사항:
  - 대상 설비의 공정 데이터를 실시간 수집할 수 있는 인프라(AAS 표준기반 체계 등)가 사전에 구축되어 있어야 함
  - 비전, 변위 등 추가 센서를 활용하면 더 정확한 분석이 가능
  - 제품 사양이 자주 바뀌는 환경에서는 제품별로 데이터를 구분해야 함
  - N-HiTS는 비정상 데이터가 연속적으로 긴 구간에서 나타나는 경우 부적합할 수 있어, 이 경우 통계 분석(가우스 혼합 + Z-score) 병행 필요
  - 불량품 발생 시 설비 이상만으로 단정하지 말고, 공정 데이터와 생산 이력(입출고 등)의 상관관계를 함께 분석해야 함


## 10. 다음 단계

이 요약을 바탕으로 `01_eda.ipynb`에서 실제 데이터를 로드하여:

- 기술통계, 결측치/단일값 컬럼 확인
- RealPower 등 주요 변수의 히스토그램/시계열 시각화
- 변수 간 상관관계(히트맵) 및 산점도 행렬
- 정상(OK) vs 비정상(NG) 테스트 데이터 비교
- `WeldingTest_03_NG_Label.csv` / `WeldingTest_04_NG_Label.csv`를 활용한 실제 이상 구간 시각화

을 수행합니다. 이후 별도 노트북에서 N-HiTS 모델 학습 및 Z-score 이상탐지를 구현할 예정입니다.
